In [1]:
library(tidyverse)

cojo_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_COJO"
res_dir  <- file.path(cojo_dir, "cojo_weak_cs/result")

weak_cs <- read_tsv(file.path(cojo_dir, "input/weak_cs_p1e-5.tsv"), show_col_types = FALSE)

## joint model: one row per CS (its top-PIP variant), all CSs of the region fitted together
joint_all <- list.files(res_dir, pattern = "cojo_joint.tsv$", recursive = TRUE, full.names = TRUE) %>%
  map_dfr(read_tsv, show_col_types = FALSE)

## stepwise selection: did slct pick a SNP inside the weak CS?
slct_all <- list.files(res_dir, pattern = "cojo_summary.tsv$", recursive = TRUE, full.names = TRUE) %>%
  map_dfr(read_tsv, show_col_types = FALSE)

slct_support <- slct_all %>%
  filter(snp_in_cs %in% TRUE) %>%
  distinct(study, block_id, cs, p_cutoff) %>%
  mutate(p_cutoff = paste0("slct_", format(p_cutoff))) %>%
  mutate(hit = TRUE) %>%
  pivot_wider(names_from = p_cutoff, values_from = hit)

## one row per weak CS
weak_joint <- weak_cs %>%
  select(study, block_id, cs, max_abs_z, p_of_maxz) %>%
  left_join(joint_all %>% select(study, block_id, cs, n_cs, SNP, PIP, marginal_z, joint_z, joint_p),
            by = c("study", "block_id", "cs")) %>%
  left_join(slct_support, by = c("study", "block_id", "cs")) %>%
  mutate(across(starts_with("slct_"), ~ replace_na(.x, FALSE))) %>%
  arrange(joint_p)

weak_joint

## counts
weak_joint %>%
  summarise(n_weak_cs        = n(),
            multi_cs_region  = sum(n_cs > 1, na.rm = TRUE),
            `joint p < 5e-8` = sum(joint_p < 5e-8, na.rm = TRUE),
            `joint p < 1e-6` = sum(joint_p < 1e-6, na.rm = TRUE),
            `joint p < 1e-5` = sum(joint_p < 1e-5, na.rm = TRUE),
            `joint p < 1e-3` = sum(joint_p < 1e-3, na.rm = TRUE),
            stronger_joint   = sum(abs(joint_z) > abs(marginal_z), na.rm = TRUE),
            across(starts_with("slct_"), sum))

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


study,block_id,cs,max_abs_z,p_of_maxz,n_cs,SNP,PIP,marginal_z,joint_z,joint_p,slct_5e-08,slct_1e-06
<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<lgl>
AD_Bellenguez_nobiobank_2026,chr11_119829118_122612025,susie_rss_2,4.247,2.17e-05,2,chr11:121525902:C:T,0.13522878,4.247009,6.081315,1.19205e-09,TRUE,TRUE
AD_Bellenguez_nobiobank_2026,chr19_53788025_55215821,susie_rss_3,4.200,2.67e-05,3,chr19:54302487:C:T,0.11434389,-4.199792,-5.933363,2.96807e-09,FALSE,FALSE
AD_Bellenguez_main_2026,chr2_125689597_127728648,susie_rss_2,2.728,6.37e-03,2,chr2:127147966:T:C,0.09995255,2.728262,5.786904,7.16949e-09,TRUE,TRUE
AD_Bellenguez_noproxy_2026,chr2_125689597_127728648,susie_rss_2,2.867,4.15e-03,2,chr2:127147966:T:C,0.13682043,2.866571,5.713950,1.10374e-08,TRUE,TRUE
AD_Wightman_Excluding23andMe_2021,chr11_119829118_122612025,susie_rss_2,4.140,3.47e-05,2,chr11:121522975:G:A,0.14869138,4.139954,5.625129,1.85360e-08,TRUE,TRUE
AD_Bellenguez_2022,chr2_125689597_127728648,susie_rss_2,2.926,3.44e-03,2,chr2:127151683:G:A,0.09355449,2.925532,5.587965,2.29764e-08,TRUE,TRUE
AD_Wightman_ExcludingUKBand23andME_2021,chr11_119829118_122612025,susie_rss_2,4.053,5.06e-05,2,chr11:121522975:G:A,0.17114667,4.052863,5.426378,5.75070e-08,FALSE,TRUE
AD_Bellenguez_EADB_2022,chr19_90949_1507536,susie_rss_2,3.033,2.42e-03,2,chr19:1041165:C:T,0.12101477,-3.033014,-5.426219,5.75604e-08,FALSE,TRUE
AD_Wightman_Excluding23andMe_2021,chr8_26225312_29469590,susie_rss_3,4.253,2.11e-05,3,chr8:27703134:C:T,0.68307391,-4.253143,-5.304694,1.12857e-07,FALSE,TRUE


n_weak_cs,multi_cs_region,joint p < 5e-8,joint p < 1e-6,joint p < 1e-5,joint p < 1e-3,stronger_joint,slct_5e-08,slct_1e-06
<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
12,11,6,10,11,12,11,5,8
